# Marketing Mix Modeling by PyMC-Marketing
- URL：https://www.pymc-marketing.io/en/stable/notebooks/mmm/mmm_example.html#mmm-example


## 環境構築

In [7]:
#%pip show pymc-marketing
%pip install -q pymc-marketing==1.1.0 

Note: you may need to restart the kernel to use updated packages.


In [9]:
import warnings

import arviz as az
import arviz_plots as azp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import seaborn as sns
import xarray as xr
from pymc_extras.prior import Prior

from pymc_marketing.mmm import GeometricAdstock, LogisticSaturation
from pymc_marketing.mmm.mmm import MMM
from pymc_marketing.mmm.transformers import geometric_adstock, logistic_saturation

warnings.filterwarnings("ignore", category=FutureWarning)

az.style.use("arviz-darkgrid")
plt.rcParams["figure.figsize"] = [12, 7]
plt.rcParams["figure.dpi"] = 100

%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

## データ作成

### *1. Date Range*

In [14]:
seed: int = sum(map(ord, "mmm"))
rng: np.random.Generator = np.random.default_rng(seed=seed)

# date range
min_date = pd.to_datetime("2020-04-01")
max_date = pd.to_datetime("2023-09-01")

df = pd.DataFrame(
    data={"date_week": pd.date_range(start=min_date, end=max_date, freq="W-MON")}
).assign(
    year=lambda x: x["date_week"].dt.year,
    month=lambda x: x["date_week"].dt.month,
    dayofyear=lambda x: x["date_week"].dt.dayofyear,
)

n = df.shape[0]
print(f"Number of observations: {n}")

Number of observations: 178


In [29]:
df

,date_week,year,month,dayofyear,tv,web
0,2020-04-06,2020,4,97,0.070450,0.0
1,2020-04-13,2020,4,104,0.038848,0.0
2,2020-04-20,2020,4,111,0.087786,0.0
3,2020-04-27,2020,4,118,0.414669,0.0
4,2020-05-04,2020,5,125,0.004702,0.0
...,...,...,...,...,...,...
173,2023-07-31,2023,7,212,0.111246,0.0
174,2023-08-07,2023,8,219,0.182388,0.0
175,2023-08-14,2023,8,226,0.296962,0.0
176,2023-08-21,2023,8,233,0.004671,0.0


### *2. Media Costs Data*

- 備忘録
    - to_xarray() は、Pandasのデータ（SeriesやDataFrame）を xarray.DataArray という「軸（次元）の名前やラベル情報を持った多次元配列」に変換する関数。`geometric_adstock` では引数 `dim` でindexの名前を求められるため、`to_numpy` ではエラーが起きる。

In [44]:
# media data
tv = rng.uniform(low=0.0, high=1.0, size=n)
df["tv"] = np.where(tv > 0.9, tv, tv / 2)

web = rng.uniform(low=0.0, high=1.0, size=n)
df["web"] = np.where(web > 0.8, web, 0)

# adostock parameters
alpha1: float = 0.5
alpha2: float = 0.2

# TVとWebデータに対してアドストック効果に基づく返還を適用
df["tv_adstock"] = (
    #教科書では".to_numpy()"を使用しているが、最新版docでは".to_xarray()"
    geometric_adstock(x=df["tv"].to_xarray(),
                      alpha=alpha1,
                      l_max=8,
                      dim="index",
                      normalize=True)
    .eval()
    .flatten())

df["web_adstock"] = (
    geometric_adstock(x=df["web"].to_xarray(),
                      alpha=alpha2,
                      l_max=8,
                      dim="index",
                      normalize=True)
    .eval()
    .flatten())

# 形状パラメータを設定
lam1: float = 4.0
lam2: float = 3.0

# TVとwebのアドストックデータに対して形状効果に基づく返還を適用
df["tv_adstock_saturated"] = logistic_saturation(x=df["tv_adstock"].to_xarray(),
                                                lam=lam1).eval()

df["web_adstock_saturated"] = logistic_saturation(x=df["web_adstock"].to_xarray(),
                                                  lam=lam2).eval()

# トレンド項をデータフレームに追加
df["trend"] = (np.linspace(start=0.0, stop=50, num=n) +10) ** (1 / 4) - 1

# 季節項の計算を行い、データフレームに追加
df["cs"] = -np.sin(2 * 2 * np.pi * df["dayofyear"] / 365.5)
df["cc"] = np.cos(1 * 2 * np.pi * df["dayofyear"] / 365.5)
df["seasonality"] = 0.5 * df["cs"] + df["cc"]

# イベント項、切片、誤差項をデータフレームに追加
df["event_1"] = (df["date_week"] == "2021-05-13").astype(float)
df["event_2"] = (df["date_week"] == "2022-09-14").astype(float)

df["intercept"] = 2.0
df["epsilon"] = rng.normal(loc=0.0, scale=0.25, size=n) #誤差項

# TVとWebの効果の係数をサンプルデータ生成のために設定
beta_1 = 3.0
beta_2 = 2.0
betas = [beta_1, beta_2]

# 売上データの作成
df["y"] = df["intercept"]\
    + df["trend"]\
    + df["seasonality"]\
    + 1.5 * df["event_1"]\
    + 2.5 * df["event_2"]\
    + beta_1 * df["tv_adstock_saturated"]\
    + beta_2 * df["web_adstock_saturated"]\
    + df["epsilon"]

- メディアの中での売上シェアを確認

In [48]:
contribution_share_tv: float = (beta_1 * df["tv_adstock_saturated"].sum()) / (
    beta_1 * df["tv_adstock_saturated"] + beta_2 * df["web_adstock_saturated"]).sum()

contribution_share_web: float = (beta_2 * df["web_adstock_saturated"]).sum() / (
    beta_1 * df["tv_adstock_saturated"] + beta_2 * df["web_adstock_saturated"]).sum()

print(f"Contribution Share of tv: {contribution_share_tv:.2f}")
print(f"Contribution Share of web: {contribution_share_web:.2f}")

Contribution Share of tv: 0.81
Contribution Share of web: 0.19


- ROASの確認

In [49]:
roas_tv = (beta_1 * df["tv_adstock_saturated"]).sum() / df["tv"].sum()
roas_web = (beta_2 * df["web_adstock_saturated"]).sum() / df["web"].sum()
print(f"Roas of tv: {roas_tv:.2f}")
print(f"Roas of web: {roas_web:.2f}")

Roas of tv: 5.04
Roas of web: 2.28


- データとして残すcolumnsを記載

In [52]:
columns_to_keep = [
    "date_week",
    "y",
    "tv",
    "web",
    "event_1",
    "event_2",
    "dayofyear",
]

data = df[columns_to_keep].copy()

data["t"] = range(n)

- MMMの学習時に使用する目的変数と説明変数を設定

In [53]:
X = data.drop("y", axis=1)
y = data["y"]

- 合計の広告費の中からTVとWebが占める割合を計算

In [56]:
total_spend_per_channel = data[["tv", "web"]].sum(axis=0)
spend_share = total_spend_per_channel / total_spend_per_channel.sum()
spend_share

tv     0.653169
web    0.346831
dtype: float64

- 事前分布の設定